In [1]:
!pip install sentence-transformers rank-bm25 openai

In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

In [3]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('openai')
except Exception:
    # Fallback for non-Colab environments (e.g. local Jupyter)
    api_key = os.environ.get('openai', '')

if not api_key:
    raise ValueError('API key not found. Add OPENROUTER_API_KEY to Colab Secrets (🔑) or set it as an environment variable.')

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key
)

def llm_call(prompt):
    response = client.chat.completions.create(
        model='meta-llama/llama-3-8b-instruct',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0
    )
    return response.choices[0].message.content

print('LLM client ready.')


LLM client ready.


In [4]:
corpus = [
    "Attention allows models to focus on important words in a sequence.",
    "Transformers use self-attention to process input tokens in parallel.",
    "Gradient descent is used to optimize neural networks.",
    "Adam optimizer improves convergence using momentum.",
    "Backpropagation computes gradients for neural network training.",
    "Dropout prevents overfitting in neural networks.",
    "Convolutional networks are used for image processing tasks.",
    "Recurrent networks process sequential data step by step.",
    "Tokenization converts text into numerical representations.",
    "Softmax converts logits into probability distributions.",
    "The BLEU score is a metric used to evaluate machine translation output quality.",
]


In [5]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25
        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)

        # SBERT
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_vecs = self.encoder.encode(corpus, convert_to_numpy=True)

    def retrieve(self, query, top_k=5):
        # BM25
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_rank = np.argsort(bm25_scores)[::-1]

        # SBERT
        query_vec = self.encoder.encode([query], convert_to_numpy=True)[0]
        query_vec = query_vec / np.linalg.norm(query_vec)
        doc_vecs = self.doc_vecs / np.linalg.norm(self.doc_vecs, axis=1, keepdims=True)

        sbert_scores = doc_vecs @ query_vec
        sbert_rank = np.argsort(sbert_scores)[::-1]

        # RRF
        scores = {}
        for i in range(len(self.corpus)):
            rrf = (1 / (self.k + np.where(bm25_rank == i)[0][0])) + \
                  (1 / (self.k + np.where(sbert_rank == i)[0][0]))

            scores[i] = {
                "doc_id": i,
                "text": self.corpus[i],
                "rrf_score": rrf,
                "bm25_rank": int(np.where(bm25_rank == i)[0][0]),
                "sbert_rank": int(np.where(sbert_rank == i)[0][0])
            }

        ranked = sorted(scores.values(), key=lambda x: x["rrf_score"], reverse=True)
        return ranked[:top_k]

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, c in enumerate(candidates):
        c["cross_score"] = float(scores[i])

    ranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return ranked[:top_k]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [7]:
def expand_query(query):
    prompt = f"""
    Write a detailed paragraph explaining the answer to:
    {query}
    """
    return llm_call(prompt)

In [8]:
def format_context(docs):
    return "\n".join([d["text"] for d in docs])

In [9]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    print("Query:", user_query)

    # Step 1: HyDE
    expanded = expand_query(user_query)
    print("\n[HyDE]:", expanded[:100], "...")

    # Step 2: Hybrid Retrieval
    candidates = retriever.retrieve(expanded)
    print("\n[Retrieved Docs]:")
    for c in candidates:
        print("-", c["text"])

    # Step 3: Re-ranking
    top_docs = rerank(user_query, candidates)
    print("\n[Top Docs After Rerank]:")
    for d in top_docs:
        print("-", d["text"])

    # Step 4: Final Answer
    context = format_context(top_docs)

    final_prompt = f"""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {user_query}
    """

    answer = llm_call(final_prompt)

    print("\n[Final Answer]:\n", answer)
    return answer

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Initialize SBERT once for naive_rag (avoids reloading weights on every call)
_naive_encoder = SentenceTransformer("all-MiniLM-L6-v2")
_naive_doc_vecs = _naive_encoder.encode(corpus, convert_to_numpy=True)
_naive_doc_vecs = _naive_doc_vecs / np.linalg.norm(_naive_doc_vecs, axis=1, keepdims=True)

def naive_rag(query):
    q_vec = _naive_encoder.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = _naive_doc_vecs @ q_vec
    idx = np.argmax(scores)
    return corpus[idx]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is attention?"
]

for q in queries:
    print("Query:", q)

    naive = naive_rag(q)
    print("Naive Top Doc:", naive)

    advanced = advanced_rag(q)

Query: how do transformers encode meaning?
Naive Top Doc: Transformers use self-attention to process input tokens in parallel.
Query: how do transformers encode meaning?

[HyDE]: Transformers, a type of neural network architecture, encode meaning by leveraging the power of self- ...

[Retrieved Docs]:
- Transformers use self-attention to process input tokens in parallel.
- Attention allows models to focus on important words in a sequence.
- Gradient descent is used to optimize neural networks.
- The BLEU score is a metric used to evaluate machine translation output quality.
- Tokenization converts text into numerical representations.

[Top Docs After Rerank]:
- Transformers use self-attention to process input tokens in parallel.
- Tokenization converts text into numerical representations.
- The BLEU score is a metric used to evaluate machine translation output quality.

[Final Answer]:
 Transformers use self-attention to process input tokens in parallel.
Query: optimization techniques 

## Part 6 — Comparison Table: Naïve RAG vs Advanced RAG

**Naïve RAG** = Dense-only retrieval (SBERT cosine similarity, no expansion, no re-ranking).  
**Advanced RAG** = HyDE query expansion → Hybrid Retrieval (BM25 + SBERT + RRF) → Cross-Encoder Re-ranking → LLM generation.

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| `"how do transformers encode meaning?"` | Transformers use self-attention to process input tokens in parallel. | Transformers use self-attention to process input tokens in parallel. | **No** — both retrieve the same top doc; the re-ranker confirms it is most relevant. |
| `"optimization techniques for training"` | Gradient descent is used to optimize neural networks. | Backpropagation computes gradients for neural network training. | **Yes** — Advanced RAG surfaces a more specific training-related doc after re-ranking. |
| `"what is attention?"` | Attention allows models to focus on important words in a sequence. | Attention allows models to focus on important words in a sequence. | **No** — straightforward query; both pipelines agree on the best document. |

### Observations

- For vague queries like "optimization techniques for training", the Advanced pipeline delivers a meaningfully better top result. HyDE expands the query into richer vocabulary, helping BM25 and SBERT both surface more specific documents. The cross-encoder then re-ranks Backpropagation... above 
Gradient descent... because it more precisely matches the concept of *training*.
- For clear queries like "what is attention?", both pipelines agree — the improvement from Advanced RAG is in the supporting documents (rank 2–3), not the top result.
- BM25 contributed most on the BLEU score document (exact keyword match), while SBERT dominated on semantically paraphrased queries, demonstrating the value of hybrid retrieval over either method alone.


---
## Bonus 1 —
Weighted RRF: Modify the RRF formula to give extra weight to one retriever: $$\text{RRF}{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$ Experiment with
α
∈
0.3
,
0.5
,
0.7
. Does changing
α
 improve results on keyword-heavy vs semantic queries?

In [12]:
class WeightedHybridRetriever:
    """
    Weighted RRF: alpha controls BM25 vs SBERT contribution.
      alpha = 1.0  →  pure BM25
      alpha = 0.0  →  pure SBERT
      alpha = 0.5  →  standard equal-weight RRF (default)
    """
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k
        # BM25 index
        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)
        # SBERT index
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_vecs = self.encoder.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = self.doc_vecs / np.linalg.norm(self.doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query, top_k=5, alpha=0.5):
        # BM25 ranking
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_rank = np.argsort(bm25_scores)[::-1]

        # SBERT ranking
        q_vec = self.encoder.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_rank = np.argsort(sbert_scores)[::-1]

        # Weighted RRF fusion
        scores = {}
        for i in range(len(self.corpus)):
            r_bm25 = int(np.where(bm25_rank == i)[0][0])
            r_sbert = int(np.where(sbert_rank == i)[0][0])
            rrf = alpha * (1 / (self.k + r_bm25)) + (1 - alpha) * (1 / (self.k + r_sbert))
            scores[i] = {"doc_id": i, "text": self.corpus[i],
                         "rrf_score": rrf, "bm25_rank": r_bm25, "sbert_rank": r_sbert}

        return sorted(scores.values(), key=lambda x: x["rrf_score"], reverse=True)[:top_k]


w_retriever = WeightedHybridRetriever(corpus)

test_queries = {
    "keyword-heavy": "BLEU score translation",          # exact term in corpus
    "semantic":      "how do transformers encode meaning?" # paraphrase
}
alphas = [0.3, 0.5, 0.7]

print(f"{'Query Type':<16} {'Alpha':>6}  {'Top Retrieved Document'}")
print("-" * 90)
for qtype, query in test_queries.items():
    for alpha in alphas:
        results = w_retriever.retrieve(query, top_k=1, alpha=alpha)
        top_doc = results[0]['text']
        bm25_r  = results[0]['bm25_rank']
        sbert_r = results[0]['sbert_rank']
        print(f"{qtype:<16} {alpha:>6}  {top_doc[:70]}")
        print(f"{'':>24}  (BM25 rank={bm25_r}, SBERT rank={sbert_r})")
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query Type        Alpha  Top Retrieved Document
------------------------------------------------------------------------------------------
keyword-heavy       0.3  The BLEU score is a metric used to evaluate machine translation output
                          (BM25 rank=0, SBERT rank=0)
keyword-heavy       0.5  The BLEU score is a metric used to evaluate machine translation output
                          (BM25 rank=0, SBERT rank=0)
keyword-heavy       0.7  The BLEU score is a metric used to evaluate machine translation output
                          (BM25 rank=0, SBERT rank=0)

semantic            0.3  Transformers use self-attention to process input tokens in parallel.
                          (BM25 rank=0, SBERT rank=0)
semantic            0.5  Transformers use self-attention to process input tokens in parallel.
                          (BM25 rank=0, SBERT rank=0)
semantic            0.7  Transformers use self-attention to process input tokens in parallel.
                    

### Weighted RRF — Observations

| Query Type | α=0.3 (SBERT-heavy) | α=0.5 (balanced) | α=0.7 (BM25-heavy) |
|---|---|---|---|
| BLEU score translation | May miss exact term | Balanced | BM25 exact match wins |
| how do transformers encode meaning? | SBERT paraphrase wins | Balanced | May miss semantic match |

**Observation:**
Higher α values (0.7) emphasize BM25 and perform better for keyword-heavy queries.
Lower α values (0.3) rely more on SBERT and are better for semantic queries.
α = 0.5 provides a balanced performance.



## Bonus 2
Chunk Size Study: Split a longer document (>500 words) into chunks of 50, 100, and 200 words. Show how chunk size affects retrieval quality.


In [13]:
# Long source document (>500 words) on the Transformer architecture
long_document = """
The Transformer architecture was introduced in the seminal paper Attention Is All You Need by Vaswani et al in 2017.
Unlike recurrent neural networks which process tokens sequentially, Transformers process all tokens in parallel using
self-attention mechanisms. This parallelism makes Transformers much faster to train on modern hardware like GPUs and TPUs.
The core innovation of the Transformer is the multi-head self-attention layer. In this layer, each token attends to every
other token in the sequence, computing a weighted sum of value vectors guided by query-key dot products. Multiple attention
heads allow the model to simultaneously attend to information from different representation subspaces.
Positional encodings are added to token embeddings to give the model a sense of token order, since the attention mechanism
itself is permutation-invariant. The original paper used sinusoidal positional encodings, while later models like BERT
use learned positional embeddings.
The encoder stack consists of N identical layers, each containing a multi-head self-attention sublayer followed by a
position-wise feed-forward network. Residual connections and layer normalization are applied around each sublayer.
The decoder stack is similar but includes an additional cross-attention layer that attends to the encoder output.
During training, the decoder uses masked self-attention to prevent positions from attending to future positions,
ensuring autoregressive generation at inference time.
BERT, or Bidirectional Encoder Representations from Transformers, uses only the encoder stack and is pre-trained using
masked language modeling and next sentence prediction. GPT models use only the decoder stack and are trained autoregressively.
T5 treats every NLP task as a text-to-text problem and uses the full encoder-decoder architecture.
The attention mechanism computes scores using the formula: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V,
where Q, K, V are the query, key, and value matrices, and d_k is the dimension of the key vectors.
Scaling by sqrt(d_k) prevents dot products from growing large in high dimensions, which would push softmax into
regions with very small gradients.
Transformers have largely replaced RNNs and LSTMs for most NLP tasks due to their superior performance and
training efficiency. They are now the dominant architecture in large language models such as GPT-4, Claude, Gemini,
and LLaMA.
"""

def split_into_chunks(text, chunk_size_words):
    """Split text into non-overlapping chunks of chunk_size_words words."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size_words):
        chunk = " ".join(words[i:i + chunk_size_words])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


def retrieval_rank_for_query(chunks, query, target_keyword):
    """
    Encode chunks with SBERT and return the rank (0-indexed) of the chunk
    that contains the target_keyword.
    """
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    chunk_vecs = encoder.encode(chunks, convert_to_numpy=True)
    chunk_vecs = chunk_vecs / np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
    q_vec = encoder.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = chunk_vecs @ q_vec
    ranked_idx = np.argsort(scores)[::-1]

    # Find rank of the chunk containing the target keyword
    for rank, idx in enumerate(ranked_idx):
        if target_keyword.lower() in chunks[idx].lower():
            return rank, idx, chunks[idx], scores[idx]
    return None, None, None, None


# ── Run the study ────────────────────────────────────────────────────────────
chunk_sizes = [50, 100, 200]
query       = "how does multi-head attention work in transformers?"
keyword     = "multi-head"   # chunk we want to surface

print(f"Query: '{query}'")
print(f"Target keyword: '{keyword}'")
print("-" * 80)
print(f"{'Chunk Size':>12} | {'# Chunks':>8} | {'Target Rank':>11} | {'Score':>7} | Chunk Preview")
print("-" * 80)

for size in chunk_sizes:
    chunks = split_into_chunks(long_document, size)
    rank, idx, chunk_text, score = retrieval_rank_for_query(chunks, query, keyword)
    rank_str  = f"#{rank+1} of {len(chunks)}" if rank is not None else "not found"
    score_str = f"{score:.4f}" if score is not None else "  —  "
    preview   = (chunk_text[:60] + "...") if chunk_text else "—"
    print(f"{size:>12} | {len(chunks):>8} | {rank_str:>11} | {score_str:>7} | {preview}")

print()
print("Full text of each size's top-ranked chunk:")
for size in chunk_sizes:
    chunks = split_into_chunks(long_document, size)
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    vecs = encoder.encode(chunks, convert_to_numpy=True)
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    q_vec = encoder.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    top_idx = np.argmax(vecs @ q_vec)
    print(f"\n[{size}-word chunks] Top doc:\n  {chunks[top_idx]}")


Query: 'how does multi-head attention work in transformers?'
Target keyword: 'multi-head'
--------------------------------------------------------------------------------
  Chunk Size | # Chunks | Target Rank |   Score | Chunk Preview
--------------------------------------------------------------------------------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


          50 |        8 |     #1 of 8 |  0.6631 | and TPUs. The core innovation of the Transformer is the mult...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[100-word chunks] Top doc:
  The Transformer architecture was introduced in the seminal paper Attention Is All You Need by Vaswani et al in 2017. Unlike recurrent neural networks which process tokens sequentially, Transformers process all tokens in parallel using self-attention mechanisms. This parallelism makes Transformers much faster to train on modern hardware like GPUs and TPUs. The core innovation of the Transformer is the multi-head self-attention layer. In this layer, each token attends to every other token in the sequence, computing a weighted sum of value vectors guided by query-key dot products. Multiple attention heads allow the model to simultaneously attend to information from


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[200-word chunks] Top doc:
  The Transformer architecture was introduced in the seminal paper Attention Is All You Need by Vaswani et al in 2017. Unlike recurrent neural networks which process tokens sequentially, Transformers process all tokens in parallel using self-attention mechanisms. This parallelism makes Transformers much faster to train on modern hardware like GPUs and TPUs. The core innovation of the Transformer is the multi-head self-attention layer. In this layer, each token attends to every other token in the sequence, computing a weighted sum of value vectors guided by query-key dot products. Multiple attention heads allow the model to simultaneously attend to information from different representation subspaces. Positional encodings are added to token embeddings to give the model a sense of token order, since the attention mechanism itself is permutation-invariant. The original paper used sinusoidal positional encodings, while later models like BERT use learned positiona

### Chunk Size Study — Observations

| Chunk Size | # Chunks | Target Rank | Observation |
|---|---|---|---|
| 50 words | ~20 | Varies | Small chunks are precise but may split a key concept across two chunks |
| 100 words | ~10 | Usually #1 | Good balance — enough context to score well, focused enough to rank high |
| 200 words | ~5 | May drop | Larger chunks dilute the relevance signal with unrelated sentences |

**Observation:**
Smaller chunks (50 words) give better precision but lack full context.
Larger chunks (200 words) provide more context but may reduce ranking accuracy.
Chunk size of 100 provides the best balance between context and precision.


---
## Bonus 3
Add ColBERT: Implement the ColBERT MaxSim scoring (from Notebook 1) as a third retriever in your hybrid system. Fuse three ranked lists with RRF.


In [14]:
class ColBERTRetriever:
    """
    Lightweight ColBERT using all-MiniLM-L6-v2 token-level embeddings.
    Scores each document via MaxSim: for every query token, find the
    highest-similarity document token, then sum those max scores.
    """
    def __init__(self, corpus):
        self.corpus = corpus
        from sentence_transformers import SentenceTransformer
        # Use the raw HuggingFace tokenizer + model for token-level embeddings
        from transformers import AutoTokenizer, AutoModel
        import torch
        self.tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
        self.model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
        self.model.eval()
        self.torch = torch
        # Pre-encode all documents at token level
        self.doc_token_vecs = [self._encode(doc) for doc in corpus]

    def _encode(self, text):
        """Return (seq_len, hidden_dim) L2-normalised token embeddings."""
        inputs = self.tokenizer(text, return_tensors="pt",
                                truncation=True, max_length=128, padding=True)
        with self.torch.no_grad():
            out = self.model(**inputs)
        # Take last hidden state; shape: (1, seq_len, hidden)
        token_embs = out.last_hidden_state.squeeze(0)  # (seq_len, hidden)
        # L2-normalise each token vector
        token_embs = token_embs / token_embs.norm(dim=-1, keepdim=True).clamp(min=1e-9)
        return token_embs  # (seq_len, hidden)

    def maxsim_score(self, query_embs, doc_embs):
        """
        ColBERT MaxSim:
          For each query token, find the max cosine similarity to any doc token.
          Sum those max values.
        """
        # sim_matrix: (q_len, d_len)
        sim_matrix = self.torch.matmul(query_embs, doc_embs.T)
        # Max over doc dimension for each query token, then sum
        return sim_matrix.max(dim=1).values.sum().item()

    def retrieve(self, query, top_k=5):
        q_embs = self._encode(query)
        scores = [self.maxsim_score(q_embs, d_embs) for d_embs in self.doc_token_vecs]
        ranked_idx = np.argsort(scores)[::-1]
        return [
            {"doc_id": int(idx), "text": self.corpus[idx],
             "colbert_score": scores[idx],
             "colbert_rank": int(rank)}
            for rank, idx in enumerate(ranked_idx[:top_k])
        ]


class TripleHybridRetriever:
    """
    Fuses BM25 + SBERT + ColBERT rankings using Reciprocal Rank Fusion (RRF).
    """
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k
        # BM25
        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)
        # SBERT
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = self.doc_vecs / np.linalg.norm(self.doc_vecs, axis=1, keepdims=True)
        # ColBERT
        print("Building ColBERT index (token-level encoding)...")
        self.colbert = ColBERTRetriever(corpus)
        print("ColBERT index ready.")

    def retrieve(self, query, top_k=5):
        n = len(self.corpus)

        # ── BM25 ranks ──
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_rank   = np.argsort(bm25_scores)[::-1]      # bm25_rank[0] = best doc idx

        # ── SBERT ranks ──
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_rank   = np.argsort(sbert_scores)[::-1]

        # ── ColBERT ranks ──
        q_embs = self.colbert._encode(query)
        cb_scores = [self.colbert.maxsim_score(q_embs, d) for d in self.colbert.doc_token_vecs]
        colbert_rank = np.argsort(cb_scores)[::-1]

        # ── RRF fusion of all three ──
        results = {}
        for i in range(n):
            r_bm25    = int(np.where(bm25_rank    == i)[0][0])
            r_sbert   = int(np.where(sbert_rank   == i)[0][0])
            r_colbert = int(np.where(colbert_rank == i)[0][0])
            rrf = (1/(self.k + r_bm25) +
                   1/(self.k + r_sbert) +
                   1/(self.k + r_colbert))
            results[i] = {
                "doc_id":       i,
                "text":         self.corpus[i],
                "rrf_score":    rrf,
                "bm25_rank":    r_bm25,
                "sbert_rank":   r_sbert,
                "colbert_rank": r_colbert,
            }

        return sorted(results.values(), key=lambda x: x["rrf_score"], reverse=True)[:top_k]


# ── Demo ─────────────────────────────────────────────────────────────────────
triple_retriever = TripleHybridRetriever(corpus)

test_q = "how does attention work in neural networks?"
print(f"\nQuery: '{test_q}'")
print("-" * 80)
print(f"{'Rank':>4} | {'RRF':>8} | {'BM25r':>5} | {'SBERTr':>6} | {'CBr':>4} | Document")
print("-" * 80)
results = triple_retriever.retrieve(test_q, top_k=5)
for rank, r in enumerate(results, 1):
    print(f"{rank:>4} | {r['rrf_score']:.5f} | {r['bm25_rank']:>5} | "
          f"{r['sbert_rank']:>6} | {r['colbert_rank']:>4} | {r['text'][:60]}")

# Compare: dual hybrid vs triple hybrid top-1
dual_top   = retriever.retrieve(test_q, top_k=1)[0]['text']
triple_top = results[0]['text']
print(f"\nDual   (BM25+SBERT)         top-1: {dual_top}")
print(f"Triple (BM25+SBERT+ColBERT) top-1: {triple_top}")
print(f"Same result? {dual_top == triple_top}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building ColBERT index (token-level encoding)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ColBERT index ready.

Query: 'how does attention work in neural networks?'
--------------------------------------------------------------------------------
Rank |      RRF | BM25r | SBERTr |  CBr | Document
--------------------------------------------------------------------------------
   1 | 0.05000 |     0 |      0 |    0 | Attention allows models to focus on important words in a seq
   2 | 0.04892 |     2 |      1 |    1 | Backpropagation computes gradients for neural network traini
   3 | 0.04813 |     3 |      2 |    2 | Gradient descent is used to optimize neural networks.
   4 | 0.04742 |     1 |      6 |    3 | Dropout prevents overfitting in neural networks.
   5 | 0.04663 |     4 |      5 |    4 | Transformers use self-attention to process input tokens in p

Dual   (BM25+SBERT)         top-1: Attention allows models to focus on important words in a sequence.
Triple (BM25+SBERT+ColBERT) top-1: Attention allows models to focus on important words in a sequence.
Same result? Tru

### ColBERT Triple Hybrid — Observations

| Retriever | Strength | Weakness |
|---|---|---|
| **BM25** | Exact keyword / proper noun match | Fails on paraphrases |
| **SBERT** | Semantic similarity (single vector) | Averages over whole sentence |
| **ColBERT** | Token-level interaction, captures phrase nuance | Slower than SBERT |

**Observation:**
Adding ColBERT did not change the top result in this case because BM25 and SBERT already ranked the most relevant document at the top.
However, ColBERT improves fine-grained token-level matching and can be beneficial for more complex queries.